## Task n4: Budget Scenarios

In [0]:
%sql
-- Task n4: Precomputed Budget Scenarios (25, 50, 100, and 200 wells)
-- Uses Gamma-Poisson (Empirical Bayes) shrinkage of each well's failure rate toward its area rate,
-- joined with s5 cost_per_failure by failure_type and n2 recency/failure counts.

CREATE OR REPLACE TABLE workspace.dsmlc.gold_budget_scenarios AS
WITH well_exposure AS (
    -- Producing exposure (well-years) and area in the failure window (Jun 2025 - Oct 2026)
    SELECT
        uwi,
        MAX(field_operating_area) AS area,
        MAX(facility_name) AS facility,
        -- Exposure in well-years from runtime_frac (or producing days / 365.25), floored at 0.25 yr
        GREATEST(SUM(COALESCE(runtime_frac, hours_on_prod / 24.0, 0)) / 365.25, 0.25) AS exposure_yr
    FROM workspace.dsmlc.silver_daily
    WHERE production_date >= DATE '2025-06-01'
    GROUP BY uwi
),

cost_by_type AS (
    -- Pull median cost per failure by failure_type from s5 gold_cost_per_failure
    SELECT
        failure_type,
        median_cost_per_failure
    FROM workspace.dsmlc.gold_cost_per_failure
),

well_failures AS (
    -- Count failures and compute weighted cost per failure for each well (Jun 2025+)
    SELECT
        f.uwi,
        COUNT(*) AS failure_count,
        -- Dominant failure type on this well
        MODE(f.failure_type) AS primary_failure_type,
        MODE(COALESCE(f.cause, 'Unspecified')) AS primary_failure_cause,
        -- Average of type-level median cost per failure across this well's failure mix
        AVG(COALESCE(c.median_cost_per_failure, f.linked_job_cost + COALESCE(dt.downhole_lost_oil_m3, 0) * 6.2898 * 32.50)) AS well_cost_per_failure
    FROM workspace.dsmlc.silver_failures f
    LEFT JOIN cost_by_type c
        ON f.failure_type = c.failure_type
    LEFT JOIN workspace.dsmlc.gold_failure_downtime dt
        ON f.failure_id = dt.failure_id
    WHERE f.failure_start_date >= DATE '2025-06-01'
    GROUP BY f.uwi
),

well_base AS (
    SELECT
        e.uwi,
        COALESCE(e.area, 'Unknown') AS area,
        e.facility,
        e.exposure_yr,
        COALESCE(wf.failure_count, 0) AS failure_count,
        COALESCE(wf.failure_count, 0) / e.exposure_yr AS raw_rate_yr,
        wf.primary_failure_type,
        wf.primary_failure_cause,
        wf.well_cost_per_failure
    FROM well_exposure e
    LEFT JOIN well_failures wf
        ON e.uwi = wf.uwi
),

area_priors AS (
    -- Empirical Bayes (Gamma-Poisson) prior parameters per area
    -- Shrinks each well toward its area mean rate with prior weight beta_prior = 1.0 well-year
    -- (or method-of-moments overdispersion weight when area variance > area mean)
    SELECT
        area,
        SUM(failure_count) / SUM(exposure_yr) AS area_mean_rate_yr,
        VAR_POP(raw_rate_yr) AS area_rate_var,
        -- Prior pseudo-exposure beta (clamped between 0.5 and 2.0 well-years for numerical stability)
        LEAST(
            GREATEST(
                CASE
                    WHEN VAR_POP(raw_rate_yr) > (SUM(failure_count) / SUM(exposure_yr))
                    THEN (SUM(failure_count) / SUM(exposure_yr)) / (VAR_POP(raw_rate_yr) - (SUM(failure_count) / SUM(exposure_yr)))
                    ELSE 1.0
                END,
                0.5
            ),
            2.0
        ) AS prior_beta_yr
    FROM well_base
    GROUP BY area
),

shrunken_wells AS (
    SELECT
        wb.uwi,
        wb.area,
        wb.facility,
        wb.exposure_yr,
        wb.failure_count,
        wb.raw_rate_yr,
        ap.area_mean_rate_yr,
        -- Gamma-Poisson posterior mean failure rate: (y_i + alpha) / (t_i + beta), where alpha = area_mean * beta
        (wb.failure_count + (ap.area_mean_rate_yr * ap.prior_beta_yr))
            / (wb.exposure_yr + ap.prior_beta_yr) AS shrunken_rate_yr,
        wb.primary_failure_type,
        wb.primary_failure_cause,
        wb.well_cost_per_failure,
        CASE
            WHEN wb.primary_failure_type = 'Rods'
                THEN 'Action 1: Rods (Fatigue & Corrosion Upgrade/Inhibition)'
            WHEN wb.primary_failure_type IN ('Pump', 'Tubing')
                THEN 'Action 2: Pump & Tubing (High-Cost Workover Prevention)'
            ELSE 'Action 1: Rods (General Reliability Intervention)'
        END AS recommended_action,
        -- Assumed install action cost proxy ($34,152 Workover median from s5)
        34152.0 AS action_cost_cad
    FROM well_base wb
    INNER JOIN area_priors ap
        ON wb.area = ap.area
    WHERE wb.failure_count > 0
      AND wb.well_cost_per_failure IS NOT NULL
),

scored_wells AS (
    SELECT
        uwi,
        area,
        facility,
        failure_count,
        ROUND(exposure_yr, 2) AS exposure_yr,
        ROUND(raw_rate_yr, 2) AS raw_failure_rate_yr,
        ROUND(area_mean_rate_yr, 2) AS area_prior_rate_yr,
        ROUND(shrunken_rate_yr, 2) AS shrunken_failure_rate_yr,
        primary_failure_type,
        primary_failure_cause,
        recommended_action,
        ROUND(well_cost_per_failure, 0) AS cost_per_failure_cad,
        ROUND(action_cost_cad, 0) AS action_cost_cad,
        
        -- Annual expected gross savings at 25%, 34% (vendor case), and 50% avoided share
        ROUND(shrunken_rate_yr * well_cost_per_failure * 0.25, 0) AS gross_savings_25pct_cad,
        ROUND(shrunken_rate_yr * well_cost_per_failure * 0.34, 0) AS gross_savings_34pct_cad,
        ROUND(shrunken_rate_yr * well_cost_per_failure * 0.50, 0) AS gross_savings_50pct_cad,
        
        -- Annual expected net avoided cost (Gross Savings - Action Cost) at 25% and 50%
        ROUND((shrunken_rate_yr * well_cost_per_failure * 0.25) - action_cost_cad, 0) AS net_avoided_cost_25pct_cad,
        ROUND((shrunken_rate_yr * well_cost_per_failure * 0.50) - action_cost_cad, 0) AS net_avoided_cost_50pct_cad,
        
        -- Expected avoided cost per dollar spent (at 34% midpoint / 50% share) for greedy ranking
        ROUND((shrunken_rate_yr * well_cost_per_failure * 0.50) / action_cost_cad, 3) AS savings_per_dollar_50pct,
        
        -- Break-even payback in years for this well (at 25% and 50% avoided share)
        ROUND(action_cost_cad / NULLIF(shrunken_rate_yr * well_cost_per_failure * 0.25, 0), 2) AS breakeven_years_25pct,
        ROUND(action_cost_cad / NULLIF(shrunken_rate_yr * well_cost_per_failure * 0.50, 0), 2) AS breakeven_years_50pct,
        
        -- Greedy rank by expected avoided cost per dollar
        ROW_NUMBER() OVER (
            ORDER BY ((shrunken_rate_yr * well_cost_per_failure * 0.50) / action_cost_cad) DESC,
                     failure_count DESC
        ) AS greedy_rank
    FROM shrunken_wells
),

scenarios AS (
    SELECT * FROM VALUES
        ('Scenario 1: Top 25 Wells (~0.85M Budget)', 25),
        ('Scenario 2: Top 50 Wells (~1.71M Budget)', 50),
        ('Scenario 3: Top 100 Wells (~3.42M Budget)', 100),
        ('Scenario 4: Top 200 Wells (~6.83M Budget)', 200)
    AS t(scenario_name, max_wells)
)

SELECT
    s.scenario_name,
    s.max_wells AS scenario_well_limit,
    w.*
FROM scored_wells w
INNER JOIN scenarios s
    ON w.greedy_rank <= s.max_wells
ORDER BY s.max_wells, w.greedy_rank;